# dunnhumby "The Complete Journey" — DB Summary

SQLite データベース `dunnhumby.db` の構造確認とデータ概要の把握。

In [4]:
using SQLite
using DataFrames
using Printf
using Statistics

In [5]:
DB_PATH = raw"C:\Users\kimse\OneDrive\Jupyter_notebook\dunnhumby.db"
db = SQLite.DB(DB_PATH)
println("Connected: ", DB_PATH)

Connected: C:\Users\kimse\OneDrive\Jupyter_notebook\dunnhumby.db


## 1. テーブル一覧と行数

In [4]:
# テーブル一覧取得
tables_df = DBInterface.execute(db,
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name") |> DataFrame
table_names = tables_df.name
println("テーブル数: ", length(table_names))
for t in table_names
    println("  - ", t)
end

テーブル数: 8
  - campaign_desc
  - campaign_table
  - causal_data
  - coupon
  - coupon_redempt
  - hh_demographic
  - product
  - transaction_data


In [5]:
# 各テーブルの行数
row_counts = DataFrame(table=String[], rows=Int[])
for t in table_names
    n = DBInterface.execute(db, "SELECT COUNT(*) AS cnt FROM [$t]") |> DataFrame
    push!(row_counts, (table=t, rows=n.cnt[1]))
end
sort!(row_counts, :rows, rev=true)
row_counts

Row,table,rows
,String,Int64
1,causal_data,36786524
2,transaction_data,2595732
3,coupon,124548
4,product,92353
5,campaign_table,7208
6,coupon_redempt,2318
7,hh_demographic,801
8,campaign_desc,30


## 2. 各テーブルのスキーマ

In [6]:
for t in table_names
    info = DBInterface.execute(db, "PRAGMA table_info([$t])") |> DataFrame
    println("=" ^ 60)
    @printf("%-30s (%d columns)\n", t, nrow(info))
    println("=" ^ 60)
    for row in eachrow(info)
        @printf("  %-25s %s\n", row.name, row.type)
    end
    println()
end

campaign_desc                  (4 columns)
  DESCRIPTION               TEXT
  CAMPAIGN                  TEXT
  START_DAY                 TEXT
  END_DAY                   TEXT

campaign_table                 (3 columns)
  DESCRIPTION               TEXT
  household_key             TEXT
  CAMPAIGN                  TEXT

causal_data                    (5 columns)
  PRODUCT_ID                TEXT
  STORE_ID                  TEXT
  WEEK_NO                   TEXT
  display                   TEXT
  mailer                    TEXT

coupon                         (3 columns)
  COUPON_UPC                TEXT
  PRODUCT_ID                TEXT
  CAMPAIGN                  TEXT

coupon_redempt                 (4 columns)
  household_key             TEXT
  DAY                       TEXT
  COUPON_UPC                TEXT
  CAMPAIGN                  TEXT

hh_demographic                 (8 columns)
  AGE_DESC                  TEXT
  MARITAL_STATUS_CODE       TEXT
  INCOME_DESC               TEXT
  HOMEOWNER

## 3. 各テーブルのサンプルデータ（先頭5行）

In [7]:
for t in table_names
    println("\n", "=" ^ 60)
    println("  ", t, " — 先頭5行")
    println("=" ^ 60)
    sample = DBInterface.execute(db, "SELECT * FROM [$t] LIMIT 5") |> DataFrame
    display(sample)
end


  campaign_desc — 先頭5行


Row,DESCRIPTION,CAMPAIGN,START_DAY,END_DAY
,String,String,String,String
1,TypeB,24,659,719
2,TypeC,15,547,708
3,TypeB,25,659,691
4,TypeC,20,615,685
5,TypeB,23,646,684



  campaign_table — 先頭5行


Row,DESCRIPTION,household_key,CAMPAIGN
,String,String,String
1,TypeA,17,26
2,TypeA,27,26
3,TypeA,212,26
4,TypeA,208,26
5,TypeA,192,26



  causal_data — 先頭5行


...

Row,AGE_DESC,MARITAL_STATUS_CODE,INCOME_DESC,HOMEOWNER_DESC,HH_COMP_DESC,HOUSEHOLD_SIZE_DESC,KID_CATEGORY_DESC,household_key
,String,String,String,String,String,String,String,String
1,65+,A,35-49K,Homeowner,2 Adults No Kids,2,None/Unknown,1
2,45-54,A,50-74K,Homeowner,2 Adults No Kids,2,None/Unknown,7
3,25-34,U,25-34K,Unknown,2 Adults Kids,3,1,8
4,25-34,U,75-99K,Homeowner,2 Adults Kids,4,2,13
5,45-54,B,50-74K,Homeowner,Single Female,1,None/Unknown,16



  product — 先頭5行


Row,PRODUCT_ID,MANUFACTURER,DEPARTMENT,BRAND,COMMODITY_DESC,SUB_COMMODITY_DESC,CURR_SIZE_OF_PRODUCT
,String,String,String,String,String,String,String
1,25671,2,GROCERY,National,FRZN ICE,ICE - CRUSHED/CUBED,22 LB
2,26081,2,MISC. TRANS.,National,NO COMMODITY DESCRIPTION,NO SUBCOMMODITY DESCRIPTION,
3,26093,69,PASTRY,Private,BREAD,BREAD:ITALIAN/FRENCH,
4,26190,69,GROCERY,Private,FRUIT - SHELF STABLE,APPLE SAUCE,50 OZ
5,26355,69,GROCERY,Private,COOKIES/CONES,SPECIALTY COOKIES,14 OZ



  transaction_data — 先頭5行


Row,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
,String,String,String,String,String,String,String,String,String,String,String,String
1,2375,26984851472,1,1004906,1,1.39,364,-0.6,1631,1,0,0
2,2375,26984851472,1,1033142,1,0.82,364,0,1631,1,0,0
3,2375,26984851472,1,1036325,1,0.99,364,-0.3,1631,1,0,0
4,2375,26984851472,1,1082185,1,1.21,364,0,1631,1,0,0
5,2375,26984851472,1,8160430,1,1.5,364,-0.39,1631,1,0,0


## 4. transaction_data の概要

In [8]:
# 期間・世帯数・商品数
tx_summary = DBInterface.execute(db, """
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT household_key) AS n_households,
        COUNT(DISTINCT PRODUCT_ID) AS n_products,
        COUNT(DISTINCT STORE_ID) AS n_stores,
        MIN(CAST(WEEK_NO AS INTEGER)) AS week_min,
        MAX(CAST(WEEK_NO AS INTEGER)) AS week_max,
        COUNT(DISTINCT BASKET_ID) AS n_baskets
    FROM transaction_data
""") |> DataFrame
tx_summary

Row,n_rows,n_households,n_products,n_stores,week_min,week_max,n_baskets
,Int64,Int64,Int64,Int64,Int64,Int64,Int64
1,2595732,2500,92339,582,1,102,276484


In [9]:
# 週別トランザクション数
weekly_tx = DBInterface.execute(db, """
    SELECT
        CAST(WEEK_NO AS INTEGER) AS week,
        COUNT(*) AS n_transactions,
        COUNT(DISTINCT household_key) AS n_households,
        ROUND(SUM(CAST(SALES_VALUE AS REAL)), 2) AS total_sales
    FROM transaction_data
    GROUP BY CAST(WEEK_NO AS INTEGER)
    ORDER BY week
""") |> DataFrame
println("週数: ", nrow(weekly_tx))
first(weekly_tx, 10)

週数: 102


Row,week,n_transactions,n_households,total_sales
,Int64,Int64,Int64,Float64
1,1,1881,88,5211.16
2,2,3675,175,10821.4
3,3,4803,228,13498.2
4,4,5379,270,15966.0
5,5,7168,370,20139.8
6,6,8896,433,24923.9
7,7,8980,491,28074.0
8,8,10428,530,31793.8
9,9,10585,622,31244.2


## 5. product テーブルの概要

In [10]:
# DEPARTMENT / COMMODITY_DESC の分布
dept_dist = DBInterface.execute(db, """
    SELECT DEPARTMENT, COUNT(*) AS n_products
    FROM product
    GROUP BY DEPARTMENT
    ORDER BY n_products DESC
    LIMIT 20
""") |> DataFrame
dept_dist

Row,DEPARTMENT,n_products
,String,Int64
1,GROCERY,39021
2,DRUG GM,31529
3,PRODUCE,3118
4,COSMETICS,3011
5,NUTRITION,2914
6,MEAT,2544
7,MEAT-PCKGD,2427
8,DELI,2354
9,PASTRY,2149


In [11]:
# BRAND分布（National vs Private）
brand_dist = DBInterface.execute(db, """
    SELECT BRAND, COUNT(*) AS n_products
    FROM product
    GROUP BY BRAND
    ORDER BY n_products DESC
""") |> DataFrame
brand_dist

Row,BRAND,n_products
,String,Int64
1,National,78537
2,Private,13816


## 6. campaign / coupon の概要

In [12]:
# キャンペーン種別
camp_desc = DBInterface.execute(db, "SELECT * FROM campaign_desc") |> DataFrame
camp_desc

Row,DESCRIPTION,CAMPAIGN,START_DAY,END_DAY
,String,String,String,String
1,TypeB,24,659,719
2,TypeC,15,547,708
3,TypeB,25,659,691
4,TypeC,20,615,685
5,TypeB,23,646,684
6,TypeB,21,624,656
7,TypeB,22,624,656
8,TypeA,18,587,642
9,TypeB,19,603,635


In [13]:
# キャンペーン参加世帯数
camp_summary = DBInterface.execute(db, """
    SELECT
        COUNT(*) AS n_entries,
        COUNT(DISTINCT household_key) AS n_households,
        COUNT(DISTINCT CAMPAIGN) AS n_campaigns
    FROM campaign_table
""") |> DataFrame
camp_summary

Row,n_entries,n_households,n_campaigns
,Int64,Int64,Int64
1,7208,1584,30


In [14]:
# クーポン利用率
coupon_summary = DBInterface.execute(db, """
    SELECT
        (SELECT COUNT(DISTINCT COUPON_UPC) FROM coupon) AS n_coupon_types,
        (SELECT COUNT(*) FROM coupon_redempt) AS n_redemptions,
        (SELECT COUNT(DISTINCT household_key) FROM coupon_redempt) AS n_redeeming_hh
""") |> DataFrame
coupon_summary

Row,n_coupon_types,n_redemptions,n_redeeming_hh
,Int64,Int64,Int64
1,1135,2318,434


## 7. hh_demographic の概要

In [15]:
demo = DBInterface.execute(db, "SELECT * FROM hh_demographic") |> DataFrame
println("世帯数: ", nrow(demo))
println("\nカラム: ", names(demo))
println("\n--- AGE_DESC ---")
for row in eachrow(combine(groupby(demo, :AGE_DESC), nrow => :n))
    @printf("  %-15s %d\n", row.AGE_DESC, row.n)
end
println("\n--- INCOME_DESC ---")
for row in eachrow(combine(groupby(demo, :INCOME_DESC), nrow => :n))
    @printf("  %-20s %d\n", row.INCOME_DESC, row.n)
end

世帯数: 801

カラム: ["AGE_DESC", "MARITAL_STATUS_CODE", "INCOME_DESC", "HOMEOWNER_DESC", "HH_COMP_DESC", "HOUSEHOLD_SIZE_DESC", "KID_CATEGORY_DESC", "household_key"]

--- AGE_DESC ---
  65+             72
  45-54           288
  25-34           142
  35-44           194
  19-24           46
  55-64           59

--- INCOME_DESC ---
  35-49K               172
  50-74K               192
  25-34K               77
  75-99K               96
  Under 15K            61
  100-124K             34
  15-24K               74
  125-149K             38
  150-174K             30
  250K+                11
  175-199K             11
  200-249K             5


## 8. causal_data サンプル（巨大テーブル）

In [16]:
# causal_data は 3,680万行 — サンプルのみ確認
causal_sample = DBInterface.execute(db, """
    SELECT * FROM causal_data LIMIT 10
""") |> DataFrame
causal_sample

Row,PRODUCT_ID,STORE_ID,WEEK_NO,display,mailer
,String,String,String,String,String
1,26190,286,70,0,A
2,26190,288,70,0,A
3,26190,289,70,0,A
4,26190,292,70,0,A
5,26190,293,70,0,A
6,26190,295,70,0,A
7,26190,296,70,0,A
8,26190,297,70,0,A
9,26190,298,70,0,A


In [17]:
# display の種類
causal_display = DBInterface.execute(db, """
    SELECT display, COUNT(*) AS cnt
    FROM causal_data
    WHERE display != '0' AND display IS NOT NULL
    GROUP BY display
    ORDER BY cnt DESC
    LIMIT 20
""") |> DataFrame
println("display (non-zero):")
causal_display

display (non-zero):


Row,display,cnt
,String,Int64
1,9,2699467
2,5,2575289
3,7,2362118
4,3,2073738
5,6,1816021
6,2,1812840
7,1,1102141
8,A,713180
9,4,592985


In [18]:
# mailer の種類
causal_mailer = DBInterface.execute(db, """
    SELECT mailer, COUNT(*) AS cnt
    FROM causal_data
    WHERE mailer != '0' AND mailer IS NOT NULL
    GROUP BY mailer
    ORDER BY cnt DESC
    LIMIT 20
""") |> DataFrame
println("mailer (non-zero):")
causal_mailer

mailer (non-zero):


Row,mailer,cnt
,String,Int64
1,A,17106789
2,D,4467453
3,H,1560395
4,F,1077549
5,J,306924
6,L,301327
7,C,291059
8,X,120823
9,Z,19453


## 9. テーブル間リレーション確認

In [19]:
# transaction × product JOIN確認
join_check = DBInterface.execute(db, """
    SELECT
        t.PRODUCT_ID, p.DEPARTMENT, p.COMMODITY_DESC, p.BRAND,
        CAST(t.SALES_VALUE AS REAL) AS sales,
        CAST(t.QUANTITY AS INTEGER) AS qty
    FROM transaction_data t
    JOIN product p ON t.PRODUCT_ID = p.PRODUCT_ID
    LIMIT 10
""") |> DataFrame
join_check

Row,PRODUCT_ID,DEPARTMENT,COMMODITY_DESC,BRAND,sales,qty
,String,String,String,String,Float64,Int64
1,1004906,PRODUCE,POTATOES,Private,1.39,1
2,1033142,PRODUCE,ONIONS,National,0.82,1
3,1036325,PRODUCE,VEGETABLES - ALL OTHERS,Private,0.99,1
4,1082185,PRODUCE,TROPICAL FRUIT,National,1.21,1
5,8160430,PRODUCE,ORGANICS FRUIT & VEGETABLES,Private,1.5,1
6,826249,GROCERY,BAKED BREAD/BUNS/ROLLS,Private,1.98,2
7,1043142,DRUG GM,BROOMS AND MOPS,National,1.57,1
8,1085983,GROCERY,COOKIES/CONES,National,2.99,1
9,1102651,GROCERY,PNT BTR/JELLY/JAMS,National,1.89,1


In [20]:
# transaction × hh_demographic JOIN — デモグラ付き世帯の購買
demo_tx = DBInterface.execute(db, """
    SELECT
        d.AGE_DESC, d.INCOME_DESC, d.HH_COMP_DESC,
        COUNT(*) AS n_transactions,
        ROUND(SUM(CAST(t.SALES_VALUE AS REAL)), 2) AS total_sales
    FROM transaction_data t
    JOIN hh_demographic d ON t.household_key = d.household_key
    GROUP BY d.AGE_DESC, d.INCOME_DESC, d.HH_COMP_DESC
    ORDER BY n_transactions DESC
    LIMIT 15
""") |> DataFrame
demo_tx

Row,AGE_DESC,INCOME_DESC,HH_COMP_DESC,n_transactions,total_sales
,String,String,String,Int64,Float64
1,45-54,50-74K,Unknown,51275,144942.0
2,45-54,50-74K,2 Adults No Kids,50377,1.62676e5
3,45-54,75-99K,2 Adults No Kids,32274,1.07725e5
4,35-44,50-74K,2 Adults Kids,29013,91395.7
5,35-44,75-99K,2 Adults Kids,28329,85981.4
6,35-44,35-49K,2 Adults No Kids,27336,93518.5
7,35-44,50-74K,2 Adults No Kids,25052,83751.5
8,45-54,125-149K,2 Adults Kids,23821,84891.6
9,45-54,35-49K,2 Adults Kids,22184,69178.7


## 10. Beer類データ抽出

transaction_data × product × causal_data を結合し、Beer/Ale カテゴリのみ抽出。

**カラム定義:**
| カラム | 内容 |
|--------|------|
| day | 日付（通算日番号） |
| household_key | 世帯ID |
| brand | ブランド（MANUFACTURER ID、匿名化） |
| quantity | 購入個数 |
| unit_price | 単価（SALES_VALUE / QUANTITY） |
| promo | キャンペーンあり=1 / なし=0 |

**promo判定ロジック:** 以下のいずれかを満たす場合 promo=1
- causal_data で display ≠ 0（店頭陳列プロモ）
- causal_data で mailer ≠ 0（チラシ掲載）
- RETAIL_DISC < 0 または COUPON_DISC < 0（値引き実績）

In [ ]:
# Beer類トランザクション抽出（causal_data LEFT JOIN でプロモ情報付与）
beer_df = DBInterface.execute(db, """
    SELECT
        CAST(t.WEEK_NO AS INTEGER)          AS week_no,
        CAST(t.DAY AS INTEGER)               AS day,
        t.household_key,
        p.MANUFACTURER                       AS brand,
        CAST(t.QUANTITY AS INTEGER)           AS quantity,
        ROUND(CAST(t.SALES_VALUE AS REAL), 2) AS sales_value,
        ROUND(CAST(t.SALES_VALUE AS REAL)
            / MAX(CAST(t.QUANTITY AS INTEGER), 1), 2) AS unit_price,
        CASE
            WHEN (c.display IS NOT NULL AND c.display != '0')
              OR (c.mailer  IS NOT NULL AND c.mailer  != '0')
              OR CAST(t.RETAIL_DISC AS REAL) < 0
              OR CAST(t.COUPON_DISC AS REAL) < 0
            THEN 1 ELSE 0
        END AS promo
    FROM transaction_data t
    JOIN product p
        ON t.PRODUCT_ID = p.PRODUCT_ID
    LEFT JOIN causal_data c
        ON  t.PRODUCT_ID = c.PRODUCT_ID
        AND t.STORE_ID   = c.STORE_ID
        AND t.WEEK_NO    = c.WEEK_NO
    WHERE p.COMMODITY_DESC = 'BEERS/ALES'
      AND p.DEPARTMENT    = 'GROCERY'
    ORDER BY CAST(t.WEEK_NO AS INTEGER), day, household_key
""") |> DataFrame

@printf("Beer購買レコード数: %d
", nrow(beer_df))
first(beer_df, 10)

In [ ]:
# 基本統計
println("--- Beer DataFrame 概要 ---")
@printf("レコード数:     %d
", nrow(beer_df))
@printf("ユニーク世帯数: %d
", length(unique(beer_df.household_key)))
@printf("ブランド数:     %d
", length(unique(beer_df.brand)))
@printf("期間:           day %d ~ %d
", minimum(beer_df.day), maximum(beer_df.day))
@printf("プロモあり:     %d (%.1f%%)
",
    sum(beer_df.promo .== 1),
    100.0 * sum(beer_df.promo .== 1) / nrow(beer_df))

println("
--- 数量・単価の統計 ---")
@printf("数量  mean=%.2f  median=%.1f  max=%d
",
    mean(beer_df.quantity), median(beer_df.quantity), maximum(beer_df.quantity))
@printf("単価  mean=%.2f  median=%.2f  min=%.2f  max=%.2f
",
    mean(beer_df.unit_price), median(beer_df.unit_price),
    minimum(beer_df.unit_price), maximum(beer_df.unit_price))

In [ ]:
# ブランド別集計（上位15）
brand_summary = combine(groupby(beer_df, :brand),
    nrow => :n_transactions,
    :quantity => sum => :total_qty,
    :unit_price => mean => :avg_price,
    :promo => mean => :promo_rate,
    :household_key => (x -> length(unique(x))) => :n_households
)
sort!(brand_summary, :n_transactions, rev=true)
first(brand_summary, 15)

In [ ]:
# プロモ有無別の購買比較
promo_comp = combine(groupby(beer_df, :promo),
    nrow => :n_transactions,
    :quantity => mean => :avg_qty,
    :unit_price => mean => :avg_price,
    :household_key => (x -> length(unique(x))) => :n_households
)
promo_comp

In [ ]:
# 週×ブランド別 売上集計
beer_weekly_brand = combine(groupby(beer_df, [:week_no, :brand]),
    :sales_value => sum   => :total_sales,
    :quantity    => sum   => :total_qty,
    nrow                  => :n_transactions,
    :unit_price  => mean  => :avg_unit_price,
    :promo       => mean  => :promo_rate,
    :household_key => (x -> length(unique(x))) => :n_households
)
sort!(beer_weekly_brand, [:week_no, :brand])
@printf("週×ブランド集計: %d 行 (週数=%d, ブランド数=%d)
",
    nrow(beer_weekly_brand),
    length(unique(beer_weekly_brand.week_no)),
    length(unique(beer_weekly_brand.brand)))
first(beer_weekly_brand, 15)

In [ ]:
# 週別 全ブランド合計
beer_weekly_total = combine(groupby(beer_df, :week_no),
    :sales_value => sum   => :total_sales,
    :quantity    => sum   => :total_qty,
    nrow                  => :n_transactions,
    :promo       => mean  => :promo_rate,
    :household_key => (x -> length(unique(x))) => :n_households
)
sort!(beer_weekly_total, :week_no)
@printf("週別合計: %d 週
", nrow(beer_weekly_total))
first(beer_weekly_total, 10)

In [ ]:
# CSV出力
using CSV

# 1) トランザクション明細
CSV.write("beer_transactions.csv", beer_df)
@printf("Saved: beer_transactions.csv (%d rows)
", nrow(beer_df))

# 2) 週×ブランド別集計
CSV.write("beer_weekly_brand.csv", beer_weekly_brand)
@printf("Saved: beer_weekly_brand.csv (%d rows)
", nrow(beer_weekly_brand))

# 3) 週別合計
CSV.write("beer_weekly_total.csv", beer_weekly_total)
@printf("Saved: beer_weekly_total.csv (%d rows)
", nrow(beer_weekly_total))

## 11. dunnhumby.db へテーブル書き込み

抽出・集計した3つの DataFrame を SQLite テーブルとして保存する。

| テーブル名 | 内容 |
|-----------|------|
| beer_transactions | Beer購買明細 |
| beer_weekly_brand | 週×ブランド別集計 |
| beer_weekly_total | 週別合計 |

In [ ]:
# 既存テーブルがあれば削除して再作成
for tbl in ["beer_transactions", "beer_weekly_brand", "beer_weekly_total"]
    DBInterface.execute(db, "DROP TABLE IF EXISTS []")
end
println("Dropped existing tables (if any).")

# 1) beer_transactions
SQLite.load!(beer_df, db, "beer_transactions")
n1 = DBInterface.execute(db, "SELECT COUNT(*) AS n FROM beer_transactions") |> DataFrame
@printf("beer_transactions:  %d rows written
", n1.n[1])

# 2) beer_weekly_brand
SQLite.load!(beer_weekly_brand, db, "beer_weekly_brand")
n2 = DBInterface.execute(db, "SELECT COUNT(*) AS n FROM beer_weekly_brand") |> DataFrame
@printf("beer_weekly_brand:  %d rows written
", n2.n[1])

# 3) beer_weekly_total
SQLite.load!(beer_weekly_total, db, "beer_weekly_total")
n3 = DBInterface.execute(db, "SELECT COUNT(*) AS n FROM beer_weekly_total") |> DataFrame
@printf("beer_weekly_total:  %d rows written
", n3.n[1])

In [ ]:
# 書き込み確認: テーブル一覧を再取得
tables_after = DBInterface.execute(db,
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name") |> DataFrame
println("--- 全テーブル一覧 ---")
for t in tables_after.name
    n = DBInterface.execute(db, "SELECT COUNT(*) AS cnt FROM []") |> DataFrame
    @printf("  %-25s %10d rows
", t, n.cnt[1])
end

In [ ]:
# 書き込みデータのサンプル確認
println("=== beer_transactions (先頭5行) ===")
display(DBInterface.execute(db, "SELECT * FROM beer_transactions LIMIT 5") |> DataFrame)

println("
=== beer_weekly_brand (先頭5行) ===")
display(DBInterface.execute(db, "SELECT * FROM beer_weekly_brand LIMIT 5") |> DataFrame)

println("
=== beer_weekly_total (先頭5行) ===")
display(DBInterface.execute(db, "SELECT * FROM beer_weekly_total LIMIT 5") |> DataFrame)

## まとめ

| 項目 | 値 |
|------|----|
| DB | dunnhumby.db |
| テーブル数 | 8 |
| 購買データ | ~260万行（transaction_data） |
| プロモデータ | ~3,680万行（causal_data） |
| 世帯数 | ~2,500（デモグラ付き: 801） |
| 商品数 | ~92,000 |
| 期間 | 102週間 |
| 注意 | 全カラムTEXT型 → CAST必要、ブランド匿名化 |

In [21]:
SQLite.close(db)
println("DB closed.")

DB closed.
